# Reproducing Shukla, Rivera, Bladon & Jadhav (2026) from dandiset 001845

Paper: Shukla A, Rivera EL, Bladon JH, Jadhav SP (2026). A Novel Cooperative Behavior Assay in Rats Reveals
Distinct Social Coordination Strategies of Increasing Complexity and Social Reciprocity Deficits in a Model
of Fragile X Syndrome. J Neurosci 46(24) e0143262026, doi:10.1523/jneurosci.0143-26.2026.

Dataset: DANDI:001845, "Cooperative Behavior in WT and Fmr1 Rat Dyads", version 0.260526.1807
(doi:10.48324/dandi.001845/0.260526.1807), streamed from the archive.

Version. The paper cites the dandiset without a version. The paper's code repository
(JadhavLab/Social_CoOp_Behavior, README only) names version 0.260519.1308, an early partial upload with
1,007 of the 2,649 assets. The most recent published version, 0.260526.1807, is pinned: it holds every
subject and session the claims file describes that are in the dandiset at all, and the data check below
shows that every asset of 0.260519.1308 is in it unchanged (same blob).

Claims: `claims/001845/10.1523-jneurosci.0143-26.2026.yaml`, written from the version of record (Europe PMC
full text, main text, Methods and legends; the supplement was not available to the readers).

| Claim | Tested here |
|---|---|
| fig1e-match-rate-wt-higher | tested |
| fig1f-transition-rate-wt-higher | tested |
| fig1g-match-transition-regression | tested |
| fig1h-above-chance-late | tested |
| fig1i-performance-wt-higher | tested |
| fig1j-partner-location-predicts-choice | tested |
| figs1-mixed-pairs-impaired | blocked, data missing (no mixed pairs) |
| fig2b-opaque-barrier-lowers-performance | blocked, data missing (no sensory-deprived sessions) |
| fig2c-opaque-barrier-mixed-pairs | blocked, data missing |
| fig2e-visual-cue-learning-similar | blocked, data missing (no visual-cue task) |
| fig2f-cue-latency-wt-shorter | blocked, data missing |
| fig3c-triplets-by-contingency | tested |
| fig3d-entropy-wt-lower | tested |
| fig3e-exploration-efficiency-wt-higher | tested |
| fig3g-virtual-agent-performance-similar | blocked, data missing (no virtual-agent task) |
| fig3h-virtual-agent-efficiency-similar | blocked, data missing |
| figs2-jaccard-similarity | tested (100% and 50%; no sensory-deprived sessions) |
| fig4b-arrival-lag-wt-shorter | tested |
| fig4c-departure-lag-wt-shorter | tested |
| fig4d-dwell-time | tested |
| fig4eg-cross-correlation-wt-higher | tested |
| fig4h-peak-lag-distribution | tested |
| fig4i-peak-correlation-wt-higher | tested |
| fig5d-leading-near-half | tested |
| fig5e-leading-difference-50 | tested |
| figs3-leadership-history | blocked by the claims file, methods underspecified (readings shown) |
| fig6-marl-partner-aware | blocked, methods underspecified (simulation, no code) |
| fig7b-fmr1-more-asynchronous-matches | tested |
| fig7c-trial-duration-vs-performance | tested |
| fig7e-gaze-final-quartile | tested |
| fig7g-gaze-100-late | tested |
| fig7h-gaze-50-fmr1-follower | tested |
| fig8b-partner-history-cooperation | tested |
| fig8d-fmr1-longer-delta-t | tested |
| fig8e-glmm-partner-cooperation | tested |

The streaming load takes about four minutes; the notebook about fifteen.

In [ ]:
import numpy as np, pandas as pd, collections, time, warnings
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
import coop
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
T_START = time.time()

In [ ]:
assets = coop.list_assets()
older = coop.list_assets("0.260519.1308")
blob = {a["path"]: a["blob"] for a in assets}
same = sum(blob.get(a["path"]) == a["blob"] for a in older)
print(f"version {coop.VERSION}: {len(assets)} assets ({sum(a['path'].endswith('.nwb') for a in assets)} NWB, {sum(a['path'].endswith('.mp4') for a in assets)} videos)")
print(f"version 0.260519.1308: {len(older)} assets, of which {same} are in {coop.VERSION} with the same blob")
tab = pd.DataFrame(coop.session_table(assets))
print("NWB files per subject and contingency in the pinned version:")
print(tab.groupby(["subject", "contingency"]).size().unstack().to_string())
missing = {}
for (subj, c), d in tab.groupby(["subject", "contingency"]):
    m = sorted(set(range(1, d.session.max() + 1)) - set(d.session))
    if m:
        missing[(subj, c)] = m
print("session numbers missing from a subject's sequence:", missing)

In [ ]:
rows, files, S = coop.load()
import re
print(collections.Counter(re.sub(r"subject \S+ paired with \S+", "subject X paired with Y", f["description"]) for f in files))
print(f"streamed {len(files)} NWB files of dandiset {coop.DANDISET} version {coop.VERSION}")

In [ ]:
# Counts by subject, pair, genotype, contingency
subj = pd.DataFrame([dict(subject=f["subject"], genotype=f["genotype"], side=f["side"]) for f in files])
print("files per subject and genotype:")
print(subj.groupby(["subject", "genotype"]).size().to_string())
print()
pairs = pd.DataFrame([dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"]) for s in S])
print("pair sessions analysed (first 21 at 100%, first 35 at 50%):")
print(pairs.groupby(["genotype", "pair", "contingency"]).size().unstack().to_string())
print()
print("both files of every pair session hold identical events:", all(s["events_identical"] for s in S))
print("every pair is of one genotype:", sorted(set(s["genotype"] for s in S)))
dur = np.array([(s["t1"] - s["t0"]) / 60 for s in S])
print(f"session (epoch) length: {dur.min():.1f} to {dur.max():.1f} min, median {np.median(dur):.1f} (paper: 20 to 30 min)")
print("camera metres per pixel in every file:", sorted(set(f["camera_m_per_px"] for f in files)))
npk = np.array([len(r["poke_t"]) for s in S for r in s["rats"].values()])
print(f"pokes per rat and session: {npk.min()} to {npk.max()}, median {np.median(npk):.0f}")
nm = np.array([len(s["match_t"]) for s in S])
print(f"matched pokes per pair session: {nm.min()} to {nm.max()}, median {np.median(nm):.0f}")
allpos = np.concatenate([r["pos"]["head"] for s in S for r in s["rats"].values()])
print(f"head position (cm): x {np.nanmin(allpos[:,0]):.1f} to {np.nanmax(allpos[:,0]):.1f}, y {np.nanmin(allpos[:,1]):.1f} to {np.nanmax(allpos[:,1]):.1f}; NaN fraction {np.isnan(allpos[:,0]).mean():.3f}")
fr = np.array([1 / np.median(np.diff(r["pos_t"])) for s in S for r in s["rats"].values()])
print(f"position frame rate: {fr.min():.2f} to {fr.max():.2f} Hz (paper: 30 fps)")
inside = np.mean([(r["poke_t"].min() >= s["t0"]) & (r["poke_t"].max() <= s["t1"]) for s in S for r in s["rats"].values() if len(r["poke_t"])])
print(f"fraction of rat sessions whose pokes all fall inside the epoch: {inside:.3f}")
ndrop = sum(r["n_dropped"] for s in S for r in s["rats"].values()); nall = sum(len(r["raw_poke_t"]) for s in S for r in s["rats"].values())
print(f"pokes within 0.5 s of a poke at another well (dropped as crosstalk): {ndrop} of {nall} ({ndrop / nall:.3f})")
ch = pd.DataFrame([dict(pair="-".join(s["pair"]), c=s["contingency"], rank=s["rank"], side=side, raw=len(r["raw_vw"]) - 1, clean=len(r["vw"]) - 1, dropped=r["n_dropped"]) for s in S for side, r in s["rats"].items()])
print(f"rat sessions changed by the cleaning: {(ch.raw != ch.clean).sum()} of {len(ch)}; transitions per rat session raw max {ch.raw.max()}, cleaned max {ch.clean.max()}")
print(ch.sort_values("raw", ascending=False).head(8).to_string())
consec = sum(int(np.sum(s["match_w"][1:] == s["match_w"][:-1])) for s in S)
print("consecutive matched pokes at the same well:", consec, "of", sum(max(len(s['match_w']) - 1, 0) for s in S))
rw = pd.DataFrame([dict(c=s["contingency"], m=len(s["match_t"]), r=len(s["rewarded_t"])) for s in S]).groupby("c").sum()
rw["rewarded fraction"] = rw.r / rw.m
print(rw.to_string())

### Data checks

Position and pokes share one clock. If they do, the head is at a well at the moment of a poke at that
well and is not there at the same moment shifted by some seconds. The well centre is the median head
position at the rat's pokes on that well in that session.

In [ ]:
def dist_at(s, side, dt):
    r = s["rats"][side]
    c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
    idx = np.clip(np.searchsorted(r["pos_t"], r["poke_t"] + dt), 0, len(r["pos_t"]) - 1)
    p = r["pos"]["head"][idx]
    cc = np.array([c[w] for w in r["poke_w"]])
    return np.linalg.norm(p - cc, axis=1)

for dt in (0, 2, 5, 10, -5):
    d = np.concatenate([dist_at(s, side, dt) for s in S[::7] for side in ("left", "right")])
    print(f"poke times shifted by {dt:+d} s: median head distance from the poked well {np.nanmedian(d):.1f} cm, within 6.4 cm {np.nanmean(d < coop.RADIUS_CM):.2f}")
cent = []
for s in S:
    for side, r in s["rats"].items():
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        cent.append(dict(side=side, **{f"w{k}x": c[k][0] for k in c}, **{f"w{k}y": c[k][1] for k in c}))
cent = pd.DataFrame(cent)
print("well centres (median head position at pokes, cm), median over sessions by maze:")
print(cent.groupby("side").median().round(1).to_string())
d12 = np.hypot(cent.w1x - cent.w2x, cent.w1y - cent.w2y)
print(f"distance between wells 1 and 2: median {np.nanmedian(d12):.1f} cm (maze 80 x 80 cm)")
tv = np.concatenate([np.asarray(r["table"]["stop_time"]) - np.asarray(r["table"]["start_time"]) for s in S for r in s["rats"].values()])
print(f"dataset reward_well_visits table (position based, 20 cm): {len(tv)} visits, duration median {np.median(tv):.1f} s")

### Per pair and per rat measures

A visit is a run of consecutive pokes at one well; a transition is a visit to a well other than the
previous one (claims file, opening comment). Matches are the dataset's matched pokes.
Performance is 200 x matches / transitions of both rats (Eq. 1 as the claims file reads it).

In [ ]:
P, R = [], []
for s in S:
    mins = (s["t1"] - s["t0"]) / 60
    tr = {side: max(len(r["vw"]) - 1, 0) for side, r in s["rats"].items()}
    trr = {side: max(len(r["raw_vw"]) - 1, 0) for side, r in s["rats"].items()}
    m = len(s["match_t"])
    P.append(dict(unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"],
                  matches=m, minutes=mins, match_r1=m / mins, match_r2=m, transitions=tr["left"] + tr["right"],
                  trans_r1=(tr["left"] + tr["right"]) / mins, trans_r2=tr["left"] + tr["right"],
                  performance=200 * m / max(tr["left"] + tr["right"], 1),
                  performance_raw=200 * m / max(trr["left"] + trr["right"], 1)))
    for side, r in s["rats"].items():
        R.append(dict(unit=r["subject"], pair="-".join(s["pair"]), genotype=r["genotype"], contingency=s["contingency"],
                      rank=s["rank"], side=side, transitions=tr[side], trans_r1=tr[side] / mins, trans_r2=tr[side],
                      trans_raw_r1=trr[side] / mins, trans_raw_r2=trr[side],
                      match_r1=m / mins, match_r2=m, minutes=mins))
P = pd.DataFrame(P)
R = pd.DataFrame(R)
print(P.groupby(["contingency", "genotype"])[["matches", "transitions", "performance", "minutes"]].describe().T.round(2).to_string())

def genotype_tests(df, dv, cont):
    d = df[df.contingency == cont]
    a = coop.anova_A(d, dv)
    b = coop.anova_B(d, dv)
    means = d.groupby("genotype")[dv].mean()
    return dict(dv=dv, contingency=cont, WT=means.get("WT"), Fmr1=means.get("Fmr1"),
                A_F=a["genotype"][0], A_df=f"{a['genotype'][1]},{a['genotype'][2]}", A_p=a["genotype"][3],
                A_int_p=a["interaction"][3],
                B_F=b["genotype"][0], B_df=f"{b['genotype'][1]},{b['genotype'][2]}", B_p=b["genotype"][3],
                B_int_p=b["interaction"][3])

def per_unit_means(df, dv, cont):
    return df[df.contingency == cont].groupby(["genotype", "unit"])[dv].mean().round(3).to_dict()

## fig1e-match-rate-wt-higher

> WT pairs reach significantly higher match (coordination) rates than Fmr1 pairs under both the 100% and
> the 50% reward contingency.

Rate readings r1 (matches per minute of session) and r2 (matches per session); ANOVA readings A
(ordinary two-way, as the paper's df show) and B (mixed design).

In [ ]:
t1e = pd.DataFrame([genotype_tests(P, dv, c) for dv in ("match_r1", "match_r2") for c in ("rc100", "rc50")])
print(t1e.round(4).to_string())
for c in ("rc100", "rc50"):
    print(c, "per pair mean matches per minute:", per_unit_means(P, "match_r1", c))

**Result: blocked (methods underspecified).** WT pairs make more matches than Fmr1 pairs in both contingencies (per minute 1.31 against 0.86 at 100%, 2.09 against 1.53 at 50%). Under reading A, the ordinary two-way ANOVA whose df match the paper's (F(1,168) and F(1,280)), the genotype effect is significant in both contingencies and under both rate readings (p 0.0004 and below 0.0001 per minute; 0.0015 and 0.0006 per session). Under reading B, the mixed-design ANOVA with pair as subject, it is not (p 0.18 to 0.45, F(1,8)). The claims file blocks a claim where the two readings disagree.

## fig1f-transition-rate-wt-higher

> Fmr1 rats make fewer task-related arm transitions than WT rats (they are slower in the task), under
> both contingencies.

In [ ]:
t1f = pd.DataFrame([genotype_tests(R, dv, c) for dv in ("trans_r1", "trans_r2", "trans_raw_r1", "trans_raw_r2") for c in ("rc100", "rc50")])
print(t1f.round(5).to_string())
for c in ("rc100", "rc50"):
    print(c, "per rat mean transitions per minute:", per_unit_means(R, "trans_r1", c))

**Result: blocked (methods underspecified).** WT rats make more transitions than Fmr1 rats (per minute 1.98 against 1.49 at 100%, 3.02 against 2.47 at 50%). Reading A gives a significant genotype effect in both contingencies under both rate readings (p from below 0.0001 to 0.00055, df 1,378 and 1,630, the paper's), and so do the raw, uncleaned sequences (p 0.00006 to 0.027). Reading B, with rat as subject, does not (p 0.07 to 0.42). The readings disagree.

## fig1g-match-transition-regression

> In the 50% condition, match rate rises linearly with transition rate in both genotypes, with a steeper
> slope and a better fit in WT (beta 0.79, R2 0.74) than in Fmr1 (beta 0.64, R2 0.44).

Readings of the points: (a) pair and session, the pair's summed transitions; (b) rat and session, the
rat's transitions against its pair's matches; (c) session, pairs averaged; each under r1 and r2.

In [ ]:
rows1g = []
for reading, df in (("a", P), ("b", R)):
    for r in ("r1", "r2"):
        for g in ("WT", "Fmr1"):
            d = df[(df.contingency == "rc50") & (df.genotype == g)]
            lr = stats.linregress(d[f"trans_{r}"], d[f"match_{r}"])
            rows1g.append(dict(points=reading, rate=r, genotype=g, n=len(d), slope=lr.slope, R2=lr.rvalue ** 2, p=lr.pvalue))
for r in ("r1", "r2"):
    for g in ("WT", "Fmr1"):
        d = P[(P.contingency == "rc50") & (P.genotype == g)].groupby("rank")[[f"trans_{r}", f"match_{r}"]].mean()
        lr = stats.linregress(d[f"trans_{r}"], d[f"match_{r}"])
        rows1g.append(dict(points="c", rate=r, genotype=g, n=len(d), slope=lr.slope, R2=lr.rvalue ** 2, p=lr.pvalue))
t1g = pd.DataFrame(rows1g)
t1g["slope_in_tol"] = [abs(s - (0.79 if g == "WT" else 0.64)) <= (0.079 if g == "WT" else 0.064) for s, g in zip(t1g.slope, t1g.genotype)]
t1g["R2_in_tol"] = [abs(s - (0.74 if g == "WT" else 0.44)) <= (0.074 if g == "WT" else 0.044) for s, g in zip(t1g.R2, t1g.genotype)]
print(t1g.round(4).to_string())
print("ranges plotted: transitions per minute", P.trans_r1.min().round(2), "to", P.trans_r1.max().round(2), "; matches per minute", P.match_r1.min().round(2), "to", P.match_r1.max().round(2))
fig, ax = plt.subplots(figsize=(4, 3))
for g, col in (("WT", "k"), ("Fmr1", "r")):
    d = P[(P.contingency == "rc50") & (P.genotype == g)]
    ax.scatter(d.trans_r1, d.match_r1, s=6, c=col, label=g)
ax.set_xlabel("transitions / min (pair)"); ax.set_ylabel("matches / min"); ax.legend(); plt.show()

**Result: partially reproduced.** Under reading (b), one point per rat and session, both regressions are significant and WT has the steeper slope and better fit: per minute, WT slope 0.818 and R2 0.795, Fmr1 slope 0.649 and R2 0.684. Both slopes and the WT R2 are within tolerance; the Fmr1 R2 is not (0.684 against 0.44 ± 0.044). Under readings (a) and (c) the slopes are 0.39 to 0.47, out of tolerance. The result uses the cleaned choice sequences (an assumption, see the data check).

## fig1i-performance-wt-higher

> Cooperative performance (matches relative to transitions, Eq. 1) is significantly lower in Fmr1 pairs
> than in WT pairs under both contingencies, so their impairment is not only due to slower transitions.

In [ ]:
t1i = pd.DataFrame([genotype_tests(P, dv, c) for dv in ("performance", "performance_raw") for c in ("rc100", "rc50")])
print(t1i.round(5).to_string())
for c in ("rc100", "rc50"):
    print(c, "per pair mean performance (%):", per_unit_means(P, "performance", c))
print(f"performance range {P.performance.min():.1f} to {P.performance.max():.1f} %")

**Result: blocked (methods underspecified).** Performance is higher in WT pairs (53.9 against 46.4% at 100%, 64.2 against 58.5% at 50%). Reading A finds the genotype effect in both contingencies (p = 0.015 and 0.001); reading B does not (p = 0.40 and 0.43). With the raw sequences, reading A finds it at 100% (p = 0.016) but not at 50% (p = 0.097).

## fig1h-above-chance-late

> Both WT and Fmr1 pairs coordinate no better than chance early in training, and above chance late in
> training, by a permutation test that circularly shifts one rat's sequence of well visits.

The dataset's matches cannot be recomputed under a time shift, so matches are rebuilt from the pokes
(`coop.reconstructed_matches`: overlapping visits at corresponding wells, a 3 s hold tolerance, no
match at the well of the previous match). The agreement of the rebuilt counts with the dataset's is
printed first. Observed and shifted performance both use the rebuilt rule. Shifts are uniform over
the session length (claims file, the standard choice); 1,000 per session; p is the fraction of
shifts with performance at or above the observed one (Methods, Permutation testing).

In [ ]:
def vis(r):
    return (r["vw"], r["vs"], r["ve"])

given = np.array([len(s["match_t"]) for s in S])
fits = {}
for hold in (0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0):
    rb = np.array([coop.reconstructed_matches(vis(s["rats"]["left"]), vis(s["rats"]["right"]), hold=hold) for s in S])
    fits[hold] = np.corrcoef(rb, given)[0, 1]
    print(f"  hold {hold} s: r = {fits[hold]:.4f}, total ratio {rb.sum() / given.sum():.3f}")
HOLD = max(fits, key=fits.get)
print("hold used (largest r):", HOLD)
rebuilt = np.array([coop.reconstructed_matches(vis(s["rats"]["left"]), vis(s["rats"]["right"]), hold=HOLD) for s in S])
print(f"rebuilt against dataset matches over {len(S)} pair sessions: totals {rebuilt.sum()} and {given.sum()}, r = {np.corrcoef(rebuilt, given)[0, 1]:.3f}, median abs difference {np.median(np.abs(rebuilt - given)):.0f}")

def perm_test(s, shifted_rat, rng, n=1000):
    rats = {r["subject"]: r for r in s["rats"].values()}
    a, b = s["pair"]
    r1, r2 = (rats[b], rats[a]) if shifted_rat == "first" else (rats[a], rats[b])  # r2 is shifted
    trans = sum(max(len(r["vw"]) - 1, 0) for r in s["rats"].values())
    period = s["t1"] - s["t0"]
    obs = 200 * coop.reconstructed_matches(vis(r1), vis(r2), hold=HOLD) / max(trans, 1)
    null = np.array([200 * coop.reconstructed_matches(vis(r1), vis(r2), hold=HOLD, shift=sh, t0=s["t0"], period=period) / max(trans, 1)
                     for sh in rng.uniform(0, period, n)])
    return obs, null.mean(), np.mean(null >= obs)

rng = np.random.default_rng(1845)
idx = {(s["pair"], s["contingency"], s["rank"]): s for s in S}
H = []
for pair in sorted(set(s["pair"] for s in S)):
    for cont, rank, label in (("rc100", 1, "100% first"), ("rc100", 21, "100% last"), ("rc50", 1, "50% first"), ("rc50", 35, "50% last")):
        s = idx[(pair, cont, rank)]
        for which in ("first", "second"):
            obs, nullm, p = perm_test(s, which, rng)
            H.append(dict(pair="-".join(pair), genotype=s["genotype"], session=label, shifted=which, observed=obs, null_mean=nullm, p=p,
                          dataset_perf=P[(P.unit == "-".join(pair)) & (P.contingency == cont) & (P["rank"] == rank)].performance.iloc[0]))
H = pd.DataFrame(H)
print(H.round(3).to_string())

def judge_h(early, late):
    out = {}
    for which in ("first", "second"):
        d = H[H.shifted == which]
        e = d[d.session.isin(early)].groupby(["genotype", "pair"]).p.apply(lambda x: bool((x >= 0.05).all()))
        l = d[d.session.isin(late)].groupby(["genotype", "pair"]).p.apply(lambda x: bool((x < 0.05).all()))
        both = (e & l).groupby("genotype").sum()
        out[which] = dict(early_ns=e.groupby("genotype").sum().to_dict(), late_sig=l.groupby("genotype").sum().to_dict(), both=both.to_dict())
    return out

print("reading (a) early = 100% first, late = 50% last:", judge_h(["100% first"], ["50% last"]))
print("reading (b) early = first of each contingency, late = last of each:", judge_h(["100% first", "50% first"], ["100% last", "50% last"]))

**Result: partially reproduced.** Under reading (a) (first 100% session early, last analysed 50% session late), with the first rat shifted, 4 of 5 WT and 3 of 5 Fmr1 pairs are at chance early and above chance late; with the second rat shifted, 5 of 5 WT and 3 of 5 Fmr1 pairs. The two Fmr1 pairs that fail are FXM108-FXM109 and XFN1-XFN3, not above chance in the last 50% session (p 0.17 to 0.18 and 0.061 to 0.086). Under reading (b), the first and last session of each contingency, at most 1 pair per genotype meets both conditions. The test uses matches rebuilt from the pokes with a 6 s hold, whose counts agree with the dataset's (r = 0.994); in first sessions the rebuilt performance is higher than the dataset's (for example 39.3 against 9.8 for FXM101-FXM110).

## fig1j-partner-location-predicts-choice

> A multinomial logistic regression predicts a rat's next choice from its partner's current location
> above chance in both genotypes, and more accurately in WT than in Fmr1 rats, under both contingencies.

Per rat and contingency, one sample per transition: the next well against the partner's current well
(the well of its most recent poke). Readings of the moment: (a) when the focal rat's head leaves its
well (first frame after its last poke more than 6.4 cm from the well centre); (b) its last poke
there; (c) its first poke at the next well. Pooling: p1, sessions of the contingency pooled; p2, per
session, averaged. Real accuracy: scikit-learn `LogisticRegression()` with fivefold
`StratifiedKFold` (Methods). The 1,000-shuffle null uses the class the fitted model predicts for each
partner well, which for a single one-hot predictor is the most frequent next well in the training
folds; the agreement of this shortcut with scikit-learn on the real folds is printed. Its folds
assign each class's samples to folds in turn.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

def samples(s, side, moment):
    r = s["rats"][side]
    q = s["rats"]["right" if side == "left" else "left"]
    if len(r["vw"]) < 2 or len(q["poke_t"]) == 0:
        return np.zeros(0, int), np.zeros(0, int)
    if moment == "a":
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        tt = []
        for k in range(len(r["vw"]) - 1):
            i0 = np.searchsorted(r["pos_t"], r["ve"][k]); i1 = np.searchsorted(r["pos_t"], r["vs"][k + 1])
            d = np.linalg.norm(r["pos"]["head"][i0:i1] - c[r["vw"][k]], axis=1)
            out = np.flatnonzero(d > coop.RADIUS_CM)
            tt.append(r["pos_t"][i0 + out[0]] if len(out) else r["ve"][k])
        tt = np.array(tt)
    elif moment == "b":
        tt = r["ve"][:-1]
    else:
        tt = r["vs"][1:] - 1e-6
    j = np.searchsorted(q["poke_t"], tt, side="right") - 1
    ok = j >= 0
    return q["poke_w"][j[ok]] - 1, r["vw"][1:][ok] - 1

def sk_acc(x, y):
    if len(np.unique(y)) < 2 or np.bincount(y).max() < 5:
        return np.nan, None
    X = np.eye(3)[x]
    skf = StratifiedKFold(5)
    accs, agree = [], []
    for tr, te in skf.split(X, y):
        m = LogisticRegression().fit(X[tr], y[tr])
        accs.append(np.mean(m.predict(X[te]) == y[te]))
        agree.append(np.array_equal(m.predict(np.eye(3)), modal_pred(x[tr], y[tr])))
    return np.mean(accs), np.mean(agree)

def modal_pred(x, y):
    cnt = np.zeros((3, 3)); np.add.at(cnt, (x, y), 1)
    overall = np.argmax(np.bincount(y, minlength=3))
    return np.array([np.argmax(cnt[k]) if cnt[k].sum() else overall for k in range(3)])

def null_acc(x, y, rng, n=1000):
    """Mean accuracy over n independent shuffles of x and y, modal-class predictor, stratified folds."""
    N = len(y)
    xs = x[np.argsort(rng.random((n, N)), axis=1)]
    ys = y[np.argsort(rng.random((n, N)), axis=1)]
    oh = ys[:, :, None] == np.arange(3)
    fold = ((np.cumsum(oh, axis=1) - 1) * oh).sum(2) % 5
    cnt = np.bincount((np.arange(n)[:, None] * 45 + fold * 9 + xs * 3 + ys).ravel(), minlength=n * 45).reshape(n, 5, 3, 3)
    tot = cnt.sum(1)
    correct = np.zeros(n)
    for f in range(5):
        tr = tot - cnt[:, f]
        ov = np.argmax(tr.sum(1), axis=1)
        pred = np.where(tr.sum(2) > 0, np.argmax(tr, 2), ov[:, None])
        correct += np.take_along_axis(cnt[:, f], pred[:, :, None], 2)[:, :, 0].sum(1)
    return (correct / N).mean()

rng = np.random.default_rng(18451)
J = []
agreement = []
for s0 in [None]:
    pass
units = sorted(set((r["subject"], r["genotype"]) for s in S for r in s["rats"].values()))
for cont in ("rc100", "rc50"):
    for moment in ("a", "b", "c"):
        per = collections.defaultdict(list)
        for s in S:
            if s["contingency"] != cont:
                continue
            for side, r in s["rats"].items():
                per[r["subject"]].append(samples(s, side, moment))
        for subj, g in units:
            x = np.concatenate([a for a, b in per[subj]]); y = np.concatenate([b for a, b in per[subj]])
            acc, ag = sk_acc(x, y)
            agreement.append(ag)
            J.append(dict(subject=subj, genotype=g, contingency=cont, moment=moment, pooling="p1", n=len(y), real=acc, shuffled=null_acc(x, y, rng)))
            accs, nulls = [], []
            for a, b in per[subj]:
                if len(b) >= 10 and len(np.unique(b)) >= 2 and np.bincount(b).max() >= 5:
                    acc_s, ag_s = sk_acc(a, b)
                    accs.append(acc_s); nulls.append(null_acc(a, b, rng, n=1000))
            J.append(dict(subject=subj, genotype=g, contingency=cont, moment=moment, pooling="p2", n=len(accs), real=np.nanmean(accs), shuffled=np.mean(nulls),
                          dropped=len(per[subj]) - len(accs)))
J = pd.DataFrame(J)
ag = np.array([a for a in agreement if a is not None])
print(f"modal-class shortcut predicts the same class as scikit-learn for every partner well in {ag.mean():.3f} of real folds (pooled reading)")
T = []
for (cont, moment, pooling), d in J.groupby(["contingency", "moment", "pooling"]):
    row = dict(contingency=cont, moment=moment, pooling=pooling)
    for g in ("WT", "Fmr1"):
        dg = d[d.genotype == g]
        u = stats.mannwhitneyu(dg.real, dg.shuffled, alternative="two-sided")
        row[f"{g}_real"] = dg.real.mean(); row[f"{g}_shuf"] = dg.shuffled.mean(); row[f"{g}_U"] = u.statistic; row[f"{g}_p"] = u.pvalue
    u = stats.mannwhitneyu(d[d.genotype == "WT"].real, d[d.genotype == "Fmr1"].real, alternative="two-sided")
    row["WTvsFmr1_U"] = u.statistic; row["WTvsFmr1_p"] = u.pvalue
    T.append(row)
T1j = pd.DataFrame(T)
print(T1j.round(4).to_string())
print(J[(J.pooling == "p1") & (J.moment == "b")].pivot_table(index=["genotype", "subject"], columns="contingency", values=["real", "shuffled", "n"]).round(3).to_string())

**Result: not reproduced.** Under no reading does the claim hold. WT real accuracy is not higher than Fmr1 at 100% under any reading (Mann-Whitney p 0.12 to 0.27). Fmr1 real accuracy does not exceed the shuffled one at 100% under any reading (p 0.24 to 0.79). At 50% WT exceeds Fmr1 only under reading (b, p2) (p = 0.021). WT real accuracy exceeds its shuffled accuracy in 11 of 12 readings, and at 50% Fmr1 does under readings (a) and (b) (p 0.007 to 0.031).

## fig3c-triplets-by-contingency

> Under 100% reward rats mostly alternate between two arms, while under 50% they explore all three arms
> through the optimal 1-2-3 or 1-3-2 triplets, especially WT rats.

In [ ]:
TR = []
for s in S:
    for side, r in s["rats"].items():
        tr = coop.triplets(list(r["vw"]))
        TR.append(dict(unit=r["subject"], genotype=r["genotype"], contingency=s["contingency"], rank=s["rank"], n=len(tr),
                       alt=sum(t[0] == t[2] for t in tr), o1=sum(coop.is_optimal(t, "o1") for t in tr), o2=sum(coop.is_optimal(t, "o2") for t in tr)))
TR = pd.DataFrame(TR)
pooled = TR.groupby(["contingency", "genotype", "unit"])[["n", "alt", "o1", "o2"]].sum()
for k in ("alt", "o1", "o2"):
    pooled[f"f_{k}"] = pooled[k] / pooled.n
print(pooled.round(3).to_string())
med = pooled.groupby(["contingency", "genotype"])[["f_alt", "f_o1", "f_o2"]].median()
print(med.round(3).to_string())
for o in ("o1", "o2"):
    c1 = med.loc[("rc100", "WT"), "f_alt"] > 0.5 and med.loc[("rc100", "Fmr1"), "f_alt"] > 0.5
    c2 = med.loc[("rc50", "WT"), f"f_{o}"] > med.loc[("rc100", "WT"), f"f_{o}"]
    c3 = med.loc[("rc50", "WT"), f"f_{o}"] > med.loc[("rc50", "Fmr1"), f"f_{o}"]
    print(f"reading {o}: alternation > 0.5 at 100% in both genotypes {c1}; WT optimal 50% > 100% {c2}; WT > Fmr1 optimal at 50% {c3}")

**Result: partially reproduced.** At 50% the WT median fraction of optimal triplets exceeds its 100% value (o1 0.807 against 0.468, o2 0.269 against 0.171) and the Fmr1 median at 50% (0.807 against 0.687; 0.269 against 0.259), under both readings. At 100% the WT median alternation fraction is 0.532 but the Fmr1 median is 0.369, so the Fmr1 rats do not predominantly alternate; 4 of 10 Fmr1 rats are above 0.5.

## fig3d-entropy-wt-lower

> The entropy of WT rats' choice sequences is lower than that of Fmr1 rats, a genotype effect under both
> contingencies.

Sample entropy of each rat's session sequence of visited wells, m = 2 and m = 3, r = 0.2 x SD. A
session with no template match of length m + 1 has undefined sample entropy and is dropped (counted).

In [ ]:
EN = []
for s in S:
    for side, r in s["rats"].items():
        x = np.asarray(r["vw"], float)
        sd = x.std() if len(x) > 1 else 0
        row = dict(unit=r["subject"], genotype=r["genotype"], contingency=s["contingency"], rank=s["rank"])
        for m in (2, 3):
            row[f"se_m{m}"] = coop.sample_entropy(x, m, 0.2 * sd) if sd > 0 else np.nan
        EN.append(row)
EN = pd.DataFrame(EN)
print("undefined (dropped) values:", EN[["se_m2", "se_m3"]].isna().sum().to_dict(), "of", len(EN))
t3d = pd.DataFrame([genotype_tests(EN, f"se_m{m}", c) for m in (2, 3) for c in ("rc100", "rc50")])
print(t3d.round(5).to_string())
for m in (2, 3):
    for c in ("rc100", "rc50"):
        for g in ("WT", "Fmr1"):
            d = EN[(EN.contingency == c) & (EN.genotype == g)].dropna(subset=[f"se_m{m}"])
            lr = stats.linregress(d["rank"], d[f"se_m{m}"])
            print(f"m={m} {c} {g}: slope over sessions {lr.slope:.4f} per session (p = {lr.pvalue:.3g})")
print("range:", EN.se_m2.min().round(3), "to", EN.se_m2.max().round(3), "(m=2);", EN.se_m3.min().round(3), "to", EN.se_m3.max().round(3), "(m=3)")

**Result: partially reproduced.** At 50% WT entropy is lower than Fmr1 under every reading (m = 2: 0.250 against 0.380, reading A p below 0.0001, B p = 0.035; m = 3: 0.240 against 0.365, A p below 0.0001, B p = 0.037). At 100% the genotype effect is not significant under any reading (A p = 0.22 and 0.059, B p = 0.59 and 0.33). The genotype x session terms of reading A are not significant (p 0.24 to 0.74), as the paper reports. Entropy falls over sessions in both genotypes (slopes -0.019 to -0.0025 per session).

## fig3e-exploration-efficiency-wt-higher

> WT rats have higher exploration efficiency (fraction of optimal choice triplets) than Fmr1 rats,
> strongly in the 50% condition and modestly in the 100% condition.

In [ ]:
TR["eff_o1"] = TR.o1 / TR.n.replace(0, np.nan)
TR["eff_o2"] = TR.o2 / TR.n.replace(0, np.nan)
print("rat sessions with no triplet (dropped):", int((TR.n == 0).sum()))
rows3e = []
for o in ("o1", "o2"):
    rows3e.append(dict(genotype_tests(TR, f"eff_{o}", "rc100"), sessions=21))
    for n in (33, 35):
        rows3e.append(dict(genotype_tests(TR[(TR.contingency == "rc100") | (TR["rank"] <= n)], f"eff_{o}", "rc50"), sessions=n))
t3e = pd.DataFrame(rows3e)
print(t3e.round(6).to_string())

**Result: blocked (methods underspecified).** At 50%, with every triplet visiting all three wells counted as optimal (o1), WT efficiency is higher (0.773 against 0.692 over 35 sessions) and reading A finds it (p below 0.0001), reading B does not (p = 0.086 and 0.10). With only 1-2-3 and 1-3-2 counted (o2) there is no difference (0.273 against 0.262, A p = 0.50 and 0.76). At 100% neither reading finds an effect (A p = 0.058 and 0.34).

## figs2-jaccard-similarity

> Within pairs, choice sequences are similar for both genotypes under 100% reward, but under 50%
> reward WT pairs have more similar choice sequences than Fmr1 pairs; in the sensory-deprived condition
> similarity falls in both genotypes alike.

Every session of one rat against every session of its partner (Methods, Eq. 3). Sets: (a) distinct
transitions, (b) distinct nonoverlapping triplets, (c) distinct overlapping triplets. Summaries: (i)
all entries, (ii) lower triangle with the diagonal, (iii) diagonal. Two-sided Mann-Whitney U on the 5
and 5 pair values (the claims file's stand-in). The sensory-deprived sessions are not in the dataset.

In [ ]:
def sets(seq, kind):
    seq = list(seq)
    if kind == "a":
        return set(zip(seq[:-1], seq[1:]))
    if kind == "b":
        return set(coop.triplets(seq))
    return set(zip(seq[:-2], seq[1:-1], seq[2:]))

def jac(a, b):
    u = a | b
    return len(a & b) / len(u) if u else np.nan

JA = []
for pair in sorted(set(s["pair"] for s in S)):
    for cont in ("rc100", "rc50"):
        ss = sorted([s for s in S if s["pair"] == pair and s["contingency"] == cont], key=lambda s: s["rank"])
        g = ss[0]["genotype"]
        r1 = [{r["subject"]: r for r in s["rats"].values()}[pair[0]]["vw"] for s in ss]
        r2 = [{r["subject"]: r for r in s["rats"].values()}[pair[1]]["vw"] for s in ss]
        for kind in ("a", "b", "c"):
            A = [sets(x, kind) for x in r1]; B = [sets(x, kind) for x in r2]
            M = np.array([[jac(a, b) for a in A] for b in B])  # rows: rat 2 sessions, columns: rat 1
            JA.append(dict(pair="-".join(pair), genotype=g, contingency=cont, sets=kind, i=np.nanmean(M),
                           ii=np.nanmean(M[np.tril_indices(len(M))]), iii=np.nanmean(np.diag(M))))
JA = pd.DataFrame(JA)
rows = []
for (cont, kind), d in JA.groupby(["contingency", "sets"]):
    for summ in ("i", "ii", "iii"):
        w = d[d.genotype == "WT"][summ]; f = d[d.genotype == "Fmr1"][summ]
        u = stats.mannwhitneyu(w, f, alternative="two-sided")
        rows.append(dict(contingency=cont, sets=kind, summary=summ, WT=w.mean(), Fmr1=f.mean(), U=u.statistic, p=u.pvalue))
TJ = pd.DataFrame(rows)
print(TJ.round(4).to_string())

**Result: not reproduced.** At 50% WT similarity does not exceed Fmr1 under any reading. Under the transition and overlapping-triplet sets it is lower (for example 0.750 against 0.893, U = 2, p = 0.032, transitions, all entries), and under the nonoverlapping-triplet sets the difference is not significant (p 0.22 to 0.31). At 100% similarity is 0.35 to 0.86 in both genotypes with no difference (p 0.22 to 1.0). The sensory-deprived sessions are not in the dandiset.

## Timing at the wells (Figure 4B to 4D)

For each of the dataset's matches, each rat's visit is its latest visit at the matched well that
began before the match. Arrival: (a) the first poke of that visit; (b) the first frame of the run of
frames in which the head is within 6.4 cm (40 px) of the well centre that contains the match time.
Departure: (a) the last poke of the visit; (b) the last frame of that run. Dwell, per visit (all
visits): (a) first to last poke; (b) the length of each run of frames within 6.4 cm of a well.

In [ ]:
def head_runs(r, centres):
    occ = coop.occupancy(r["pos"]["head"], centres)
    return occ

TM, DW = [], []
for s in S:
    per_rat = {}
    for side, r in s["rats"].items():
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        occ = coop.occupancy(r["pos"]["head"], c)
        per_rat[side] = (r, occ)
        # dwell (a) poke runs, (b) runs of head occupancy
        dw_a = r["ve"] - r["vs"]
        ch = np.flatnonzero(np.diff(np.r_[0, occ, 0]) != 0)
        runs = []
        b = np.r_[0, occ, 0]
        starts = np.flatnonzero((b[1:-1] != 0) & (b[1:-1] != b[:-2]))
        ends = np.flatnonzero((b[1:-1] != 0) & (b[1:-1] != b[2:]))
        dw_b = r["pos_t"][ends] - r["pos_t"][starts]
        DW.append(dict(unit=r["subject"], genotype=r["genotype"], contingency=s["contingency"], rank=s["rank"],
                       dwell_a=dw_a.mean() if len(dw_a) else np.nan, dwell_b=dw_b.mean() if len(dw_b) else np.nan))
    arr = {"a": [], "b": []}; dep = {"a": [], "b": []}
    for t, w in zip(s["match_t"], s["match_w"]):
        ta, tb, da, db = {}, {}, {}, {}
        for side, (r, occ) in per_rat.items():
            k = np.flatnonzero((r["vw"] == w) & (r["vs"] <= t + 0.01))
            if len(k) == 0:
                break
            k = k[-1]
            ta[side], da[side] = r["vs"][k], r["ve"][k]
            i = min(np.searchsorted(r["pos_t"], t), len(occ) - 1)
            if occ[i] == w:
                i0 = i
                while i0 > 0 and occ[i0 - 1] == w:
                    i0 -= 1
                i1 = i
                while i1 < len(occ) - 1 and occ[i1 + 1] == w:
                    i1 += 1
                tb[side], db[side] = r["pos_t"][i0], r["pos_t"][i1]
        if len(ta) == 2:
            arr["a"].append(abs(ta["left"] - ta["right"])); dep["a"].append(abs(da["left"] - da["right"]))
        if len(tb) == 2:
            arr["b"].append(abs(tb["left"] - tb["right"])); dep["b"].append(abs(db["left"] - db["right"]))
    TM.append(dict(unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], matches=len(s["match_t"]),
                   arr_a=np.mean(arr["a"]) if arr["a"] else np.nan, arr_b=np.mean(arr["b"]) if arr["b"] else np.nan,
                   dep_a=np.mean(dep["a"]) if dep["a"] else np.nan, dep_b=np.mean(dep["b"]) if dep["b"] else np.nan,
                   n_a=len(arr["a"]), n_b=len(arr["b"])))
TM = pd.DataFrame(TM); DW = pd.DataFrame(DW)
print(f"matches with both arrivals by pokes: {TM.n_a.sum()} of {TM.matches.sum()}; by head position: {TM.n_b.sum()}")
print(TM.groupby(["contingency", "genotype"])[["arr_a", "arr_b", "dep_a", "dep_b"]].describe().T.round(2).to_string())
print(DW.groupby(["contingency", "genotype"])[["dwell_a", "dwell_b"]].describe().T.round(2).to_string())

## fig4b-arrival-lag-wt-shorter

> During match events WT pairs arrive at the wells with shorter lags between the two rats than Fmr1
> pairs, under both contingencies; in WT pairs the lag falls over 100% training.

In [ ]:
t4b = pd.DataFrame([genotype_tests(TM, dv, c) for dv in ("arr_a", "arr_b") for c in ("rc100", "rc50")])
print(t4b.round(5).to_string())
for dv in ("arr_a", "arr_b"):
    d = TM[(TM.contingency == "rc100") & (TM.genotype == "WT")].dropna(subset=[dv])
    lr = stats.linregress(d["rank"], d[dv])
    print(f"{dv}: WT lag over 100% sessions slope {lr.slope:.3f} s per session (p = {lr.pvalue:.3g})")
    print("  per pair means:", per_unit_means(TM, dv, "rc100"), per_unit_means(TM, dv, "rc50"))

**Result: not reproduced.** No reading gives a WT-shorter lag in both contingencies. By pokes (a), WT lags are shorter at 100% (16.6 against 23.3 s, A p = 0.018, B p = 0.19) but not at 50% (12.1 against 13.5 s, A p = 0.43). By head position (b), they are shorter at 50% (3.9 against 5.9 s, A p below 0.0001, B p = 0.011) but not at 100% (A p = 0.37; one WT pair has a mean of 43.7 s). The WT lag falls over the 100% sessions by pokes (-1.0 s per session, p = 0.00015) and not by head position (p = 0.60).

## fig4c-departure-lag-wt-shorter

> After match events WT rats depart from the wells with shorter lags between partners than Fmr1 rats,
> under both contingencies.

In [ ]:
rows = []
for dv in ("dep_a", "dep_b"):
    rows.append(dict(genotype_tests(TM, dv, "rc100"), sessions=21))
    for n in (33, 35):
        rows.append(dict(genotype_tests(TM[(TM.contingency == "rc100") | (TM["rank"] <= n)], dv, "rc50"), sessions=n))
t4c = pd.DataFrame(rows)
print(t4c.round(5).to_string())

**Result: not reproduced.** By pokes (a) there is no genotype effect in either contingency (A p = 0.17 at 100%, 0.64 and 0.71 at 50%). By head position (b) WT pairs depart with shorter lags at 50% (4.6 against 7.8 s, A p below 0.0001, B p = 0.084 and 0.086) but not at 100% (A p = 0.67).

## fig4d-dwell-time

> Fmr1 rats spend longer at reward wells per visit than WT rats (Results), though the legend's genotype
> effects are not significant (p = 0.25 at 100%, p = 0.063 at 50%).

In [ ]:
t4d = pd.DataFrame([genotype_tests(DW, dv, c) for dv in ("dwell_a", "dwell_b") for c in ("rc100", "rc50")])
print(t4d.round(5).to_string())
print("per rat mean dwell (a), 50%:", per_unit_means(DW, "dwell_a", "rc50"))

**Result: reproduced.** Fmr1 means are higher than WT means in both contingencies under both readings: by pokes 33.4 against 30.1 s at 100% and 18.3 against 16.9 s at 50%; by head position 5.25 against 5.04 s and 5.352 against 5.344 s. The genotype effect of reading A is not significant (p = 0.21 and 0.35 by pokes, 0.45 and 0.97 by head position), as the legend reports.

## Cross-correlation of well arrivals (Figure 4E to 4I)

Occupancy: the nearest well within 6.4 cm of the head, else null; 1 Hz by (f) the frame at each
second or (m) the modal value in each second; arrivals where the well differs from the previous
second, with (y) or without (n) a return to the same well after null counted. Positive lag: rat 2
(the second subject of the pair, alphabetically) follows rat 1. Lags -20 to +20 s. The shuffle is a
circular shift of rat 2's arrival series by a uniform random amount, 20 shifts per session averaged.

In [ ]:
def xc_session(s, mode, null_return, rng):
    rats = {r["subject"]: r for r in s["rats"].values()}
    ser = []
    for subj in s["pair"]:
        r = rats[subj]
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        occ = coop.occupancy(r["pos"]["head"], c)
        ser.append(coop.arrivals(coop.downsample_occ(occ, r["pos_t"], s["t0"], s["t1"], "frame" if mode == "f" else "modal"), null_return))
    n = min(len(ser[0]), len(ser[1]))
    a1, a2 = ser[0][:n], ser[1][:n]
    lags, c = coop.xcorr(a1, a2)
    sh = np.mean([coop.xcorr(a1, np.roll(a2, int(k)), 20)[1] for k in rng.integers(1, n, 20)], axis=0)
    return lags, c, sh, a1.sum(), a2.sum()

rng = np.random.default_rng(18452)
XC, PK = [], []
for mode in ("f", "m"):
    for nr in ("y", "n"):
        for s in S:
            lags, c, sh, n1, n2 = xc_session(s, mode, nr == "y", rng)
            k = np.abs(lags) <= 20
            for L, v, vs in zip(lags[k], c[k], sh):
                XC.append(dict(reading=mode + nr, unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], lag=int(L), coef=v, shuf=vs))
            ok = np.isfinite(c)
            row = dict(reading=mode + nr, unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], arrivals1=n1, arrivals2=n2)
            if ok.any() and np.nanmax(c) > 0:
                kk = k & ok
                row.update(peak20=np.nanmax(c[kk]), lag20=abs(lags[kk][np.nanargmax(c[kk])]), peakall=np.nanmax(c), lagall=abs(lags[np.nanargmax(c)]))
            PK.append(row)
XC = pd.DataFrame(XC); PK = pd.DataFrame(PK)
print("sessions without arrivals in a rat (no correlation):", PK.groupby("reading").peak20.apply(lambda x: int(x.isna().sum())).to_dict())
print("arrivals per rat and session (reading fy):", PK[PK.reading == "fy"][["arrivals1", "arrivals2"]].describe().round(1).to_string())

## fig4eg-cross-correlation-wt-higher

> Cross-correlation of the two rats' well-arrival times has peaks above shuffled controls in both
> genotypes, but correlation is higher in WT pairs and only WT pairs show prominent peaks near zero lag.

Linear mixed model coef ~ genotype x lag (lag categorical, sum coding) with a random intercept per
pair; Wald tests of the genotype term and of the genotype x lag terms.

In [ ]:
import statsmodels.formula.api as smf

def lmm_genotype_lag(d):
    d = d.dropna(subset=["coef"]).copy()
    fit = smf.mixedlm("coef ~ C(genotype, Sum) * C(lag, Sum)", d, groups=d["unit"]).fit(reml=True)
    names = fit.fe_params.index
    out = {}
    for term, sel in (("genotype", [i for i, n in enumerate(names) if n.startswith("C(genotype, Sum)") and ":" not in n]),
                      ("genotype x lag", [i for i, n in enumerate(names) if n.startswith("C(genotype, Sum)") and ":" in n])):
        L = np.zeros((len(sel), len(fit.params)))
        for j, i in enumerate(sel):
            L[j, i] = 1
        w = fit.wald_test(L, scalar=True)
        out[term] = (float(w.statistic) / len(sel), len(sel), int(fit.nobs - len(names)), float(w.pvalue))
    return out

rows = []
for reading in ("fy", "fn", "my", "mn"):
    for c in ("rc100", "rc50"):
        d = XC[(XC.reading == reading) & (XC.contingency == c)]
        res = lmm_genotype_lag(d)
        m = d.groupby("genotype").coef.mean()
        z = d[d.lag.abs() <= 1].groupby("genotype").coef.mean()
        sh = d.groupby("genotype").shuf.mean()
        rows.append(dict(reading=reading, contingency=c, WT_mean=m["WT"], Fmr1_mean=m["Fmr1"], WT_shuf=sh["WT"], Fmr1_shuf=sh["Fmr1"],
                         WT_lag0=z["WT"], Fmr1_lag0=z["Fmr1"], F_genotype=res["genotype"][0], df_g=f"{res['genotype'][1]},{res['genotype'][2]}", p_genotype=res["genotype"][3],
                         F_int=res["genotype x lag"][0], df_int=f"{res['genotype x lag'][1]},{res['genotype x lag'][2]}", p_int=res["genotype x lag"][3]))
t4eg = pd.DataFrame(rows)
print(t4eg.round(5).to_string())
prof = XC[XC.reading == "fy"].groupby(["contingency", "genotype", "lag"])[["coef", "shuf"]].mean().unstack(["contingency", "genotype"])
print("mean coefficient by lag, reading fy (observed and shuffled):")
print(prof.round(4).to_string())
print("coefficient range:", XC.coef.min().round(4), "to", XC.coef.max().round(4))

**Result: partially reproduced.** WT coefficients are higher than Fmr1 at every reading, contingency and at lags -1 to +1 s (for example frame sampling, returns counted: 0.078 against 0.055 at 100%, 0.086 against 0.070 at 50%), and the genotype x lag interaction is significant at 50% under every reading (p below 0.0001) and at 100% under three of four (p 0.0003 to 0.046; 0.064 for modal sampling with returns). The genotype main effect is not significant at 50% under any reading (p 0.30 to 0.35) and at 100% only when returns to the same well are not counted (p = 0.009 and 0.014). Near zero lag the WT mean (0.084 at lag 0, 100%) is above its shuffled mean (0.066); the Fmr1 mean (0.056) is close to its shuffled mean (0.054).

## fig4h-peak-lag-distribution

> The distribution of lags at peak cross-correlation differs between WT and Fmr1 pairs in the 50%
> condition but not in the 100% condition.

In [ ]:
rows = []
for reading in ("fy", "fn", "my", "mn"):
    for rng_ in ("20", "all"):
        for c in ("rc100", "rc50"):
            d = PK[(PK.reading == reading) & (PK.contingency == c)].dropna(subset=[f"lag{rng_}"])
            w = d[d.genotype == "WT"][f"lag{rng_}"]; f = d[d.genotype == "Fmr1"][f"lag{rng_}"]
            ks = stats.ks_2samp(w, f)
            rows.append(dict(reading=reading, range=rng_, contingency=c, n_WT=len(w), n_Fmr1=len(f), WT_median=w.median(), Fmr1_median=f.median(), D=ks.statistic, p=ks.pvalue))
t4h = pd.DataFrame(rows)
print(t4h.round(5).to_string())

**Result: not reproduced (resting on an assumption).** The readings disagree. Under the reading the Methods describe most directly (the frame at each second, every change into a well an arrival, lags -20 to +20 s), the 50% distributions do not differ (KS p = 0.44) and neither do the 100% ones (p = 0.34). Only one reading of eight gives the claimed pattern (frame sampling, returns not counted, all lags: 100% p = 0.15, 50% p = 0.00016).

## fig4i-peak-correlation-wt-higher

> Session-wise peak cross-correlation coefficients are higher in WT than in Fmr1 pairs under both
> contingencies.

In [ ]:
rows = []
for reading in ("fy", "fn", "my", "mn"):
    for rng_ in ("20", "all"):
        for c in ("rc100", "rc50"):
            d = PK[PK.reading == reading]
            rows.append(dict(genotype_tests(d, f"peak{rng_}", c), reading=reading, range=rng_))
t4i = pd.DataFrame(rows)
print(t4i.round(5).to_string())

**Result: blocked (methods underspecified).** WT peak coefficients are higher in every reading (for example 0.142 against 0.122 at 100% and 0.149 against 0.136 at 50%, frame sampling, lags to 20 s). Reading A finds the genotype effect in both contingencies in 7 of 8 readings (p below 0.001 except one at 0.11); reading B finds it in both contingencies in 2 of 8 readings (frame sampling, returns not counted, all lags: p = 0.006 and 0.024; modal sampling, returns not counted, lags to 20 s: p = 0.021 and 0.033). The ANOVA readings disagree.

## Leaders of match events (Figure 5)

For each of the dataset's matches, the leader is the rat whose visit at the matched well began first
(first poke of the visit; Methods, Assignment of leader and follower). Ties (equal first pokes) are
counted and dropped.

In [ ]:
LEAD = []
for s in S:
    rats = {r["subject"]: r for r in s["rats"].values()}
    for t, w in zip(s["match_t"], s["match_w"]):
        st = {}
        for subj, r in rats.items():
            k = np.flatnonzero((r["vw"] == w) & (r["vs"] <= t + 0.01))
            if len(k):
                st[subj] = r["vs"][k[-1]]
        if len(st) == 2:
            a, b = s["pair"]
            if st[a] == st[b]:
                LEAD.append(dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], t=t, leader=None))
            else:
                LEAD.append(dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], t=t,
                                 leader=a if st[a] < st[b] else b, lag=abs(st[a] - st[b])))
LEAD = pd.DataFrame(LEAD)
print(f"matches with a leader: {LEAD.leader.notna().sum()} of {len(LEAD)} (ties {LEAD.leader.isna().sum()})")
LEAD = LEAD[LEAD.leader.notna()].sort_values(["pair", "contingency", "rank", "t"])

## fig5d-leading-near-half

> Leadership is not fixed; on average rats of both genotypes lead about half of the match events, with
> no genotype difference in the mean probability of leading estimated by a state-space model.

Per rat and contingency, the binary series of leading over all its match events, sessions in order,
fitted with the Smith et al. (2004) model (`coop.smith_ssm`, b0 = 0); the rat's mean p_k.

In [ ]:
SSM = []
for (pair, cont), d in LEAD.groupby(["pair", "contingency"]):
    g = d.genotype.iloc[0]
    for subj in pair.split("-"):
        n = (d.leader == subj).astype(int).values
        pk, ve = coop.smith_ssm(n)
        SSM.append(dict(subject=subj, pair=pair, genotype=g, contingency=cont, events=len(n), raw=n.mean(), ssm_mean=pk.mean(), ssm_min=pk.min(), ssm_max=pk.max(), ve=ve))
SSM = pd.DataFrame(SSM)
print(SSM.round(4).to_string())
for c in ("rc100", "rc50"):
    d = SSM[SSM.contingency == c]
    u = stats.mannwhitneyu(d[d.genotype == "WT"].ssm_mean, d[d.genotype == "Fmr1"].ssm_mean, alternative="two-sided")
    print(f"{c}: mean of rat means WT {d[d.genotype == 'WT'].ssm_mean.mean():.4f}, Fmr1 {d[d.genotype == 'Fmr1'].ssm_mean.mean():.4f}; Mann-Whitney U = {u.statistic:.1f}, p = {u.pvalue:.4f}; rat means {d.ssm_mean.min():.3f} to {d.ssm_mean.max():.3f}")

**Result: reproduced.** The genotype means of the state-space mean probability of leading are 0.500 in both genotypes and contingencies, and the Mann-Whitney tests give U = 50, p = 1.0. This follows by construction in same-genotype pairs; the rat means range from 0.290 to 0.710 at 100% and 0.315 to 0.685 at 50%.

## fig5e-leading-difference-50

> The within-pair difference in the proportion of leading events does not differ by genotype at 100%,
> but shows a small significant genotype effect at 50%, present at the start of training and gone later
> in training (Results).

In [ ]:
LD = []
for (pair, cont, rank), d in LEAD.groupby(["pair", "contingency", "rank"]):
    a = pair.split("-")[0]
    pa = (d.leader == a).mean()
    LD.append(dict(unit=pair, genotype=d.genotype.iloc[0], contingency=cont, rank=rank, absdiff=abs(2 * pa - 1), signed=2 * pa - 1))
LD = pd.DataFrame(LD)
rows = []
for dv in ("absdiff", "signed"):
    rows.append(dict(genotype_tests(LD, dv, "rc100"), sessions=21))
    for n in (33, 35):
        rows.append(dict(genotype_tests(LD[(LD.contingency == "rc100") | (LD["rank"] <= n)], dv, "rc50"), sessions=n))
t5e = pd.DataFrame(rows)
print(t5e.round(5).to_string())
for lab, sel in (("first five 50% sessions", LD["rank"] <= 5), ("last five analysed 50% sessions", LD["rank"] > 30)):
    d = LD[(LD.contingency == "rc50") & sel]
    print(lab, d.groupby("genotype").absdiff.mean().round(4).to_dict())

**Result: not reproduced.** At 50% there is no genotype effect on the within-pair difference under any reading (absolute difference, A p = 0.70 and 0.73, B p = 0.88 and 0.89; signed difference, A p = 0.061 and 0.070). At 100% there is none either (A p = 0.49), as the paper reports. In the first five 50% sessions the absolute difference is larger in WT pairs (0.397 against 0.253) and in the last five it is similar (0.271 against 0.290), which is the pattern the Results describe, without significance.

## figs3-leadership-history

> A rat's chance of leading on a given match event depends on whether it led on the preceding events.

Blocked as methods_underspecified by the claims file (the analysis is only in the unread supplement).
Shown, not judged: per rat (a) P(lead | led the previous event) against P(lead | followed it), Wilcoxon
signed-rank across rats; (b) a logistic regression of leading on the lead status of each of the
previous 1 to 3 events, pooled over rats.

In [ ]:
import statsmodels.api as sm
HS = []
for (pair, cont), d in LEAD.groupby(["pair", "contingency"]):
    for subj in pair.split("-"):
        n = (d.leader == subj).astype(int).values
        HS.append(dict(subject=subj, contingency=cont, genotype=d.genotype.iloc[0], p_after_lead=n[1:][n[:-1] == 1].mean(), p_after_follow=n[1:][n[:-1] == 0].mean()))
HS = pd.DataFrame(HS)
for c in ("rc100", "rc50"):
    d = HS[HS.contingency == c]
    w = stats.wilcoxon(d.p_after_lead, d.p_after_follow)
    print(f"{c}: P(lead | led) {d.p_after_lead.mean():.3f}, P(lead | followed) {d.p_after_follow.mean():.3f}, Wilcoxon p = {w.pvalue:.3g} (20 rats)")
    X, Y = [], []
    for (pair, cc), dd in LEAD[LEAD.contingency == c].groupby(["pair", "contingency"]):
        subj = pair.split("-")[0]
        n = (dd.leader == subj).astype(int).values
        for k in range(3, len(n)):
            X.append(n[k - 3:k][::-1]); Y.append(n[k])
    fit = sm.Logit(np.array(Y), sm.add_constant(np.array(X, float))).fit(disp=0)
    print(f"   logistic regression on the previous 1, 2, 3 events: coefficients {np.round(fit.params[1:], 3)}, p {np.array([f'{p:.2g}' for p in fit.pvalues[1:]])}")

**Result: blocked (methods underspecified), by the claims file.** Shown only: at 100% P(lead | led) is 0.473 against P(lead | followed) 0.527 (Wilcoxon p = 0.0012, 20 rats), at 50% 0.497 against 0.503 (p = 0.60); in a logistic regression the lead status two and three events back predicts leading in both contingencies (p below 1e-11).

## fig7b-fmr1-more-asynchronous-matches

> Under 50% reward, a larger proportion of Fmr1 pairs' matches occur in asynchronous trials (rats
> starting at different wells) than WT pairs' matches; at 100% there is no genotype difference.

A match is synchronous when the two rats' previous visits (the visits before the matched visits)
were at corresponding wells, and asynchronous otherwise (Results, the two trial types).

In [ ]:
AS = []
for s in S:
    nsync = nasync = 0
    for t, w in zip(s["match_t"], s["match_w"]):
        prev = []
        for r in s["rats"].values():
            k = np.flatnonzero((r["vw"] == w) & (r["vs"] <= t + 0.01))
            if len(k) and k[-1] > 0:
                prev.append(r["vw"][k[-1] - 1])
        if len(prev) == 2:
            if prev[0] == prev[1]:
                nsync += 1
            else:
                nasync += 1
    AS.append(dict(unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], sync=nsync, asyn=nasync,
                   prop_async=nasync / (nsync + nasync) if nsync + nasync else np.nan))
AS = pd.DataFrame(AS)
print(AS.groupby(["contingency", "genotype"])[["sync", "asyn"]].sum().to_string())
t7b = pd.DataFrame([genotype_tests(AS, "prop_async", c) for c in ("rc100", "rc50")])
print(t7b.round(6).to_string())
d = AS[AS.contingency == "rc50"].dropna(subset=["prop_async"])
lmm = smf.mixedlm("prop_async ~ C(genotype, Sum) * C(rank, Sum)", d, groups=d["unit"]).fit(reml=True)
names = list(lmm.fe_params.index)
L = np.zeros((1, len(lmm.params))); L[0, names.index([n for n in names if n.startswith("C(genotype, Sum)") and ":" not in n][0])] = 1
w = lmm.wald_test(L, scalar=True)
print(f"50% mixed model, genotype: F(1,{int(lmm.nobs - len(names))}) = {float(w.statistic):.3f}, p = {float(w.pvalue):.3g}")
print("per pair mean proportion asynchronous:", per_unit_means(AS, "prop_async", "rc100"), per_unit_means(AS, "prop_async", "rc50"))

**Result: partially reproduced (resting on an assumption).** At 50% a larger proportion of Fmr1 matches are asynchronous (0.363 against 0.192), and the mixed model with a random intercept per pair finds it (F(1,279) = 4.18, p = 0.041); reading A also does (p below 0.0001). At 100% the Fmr1 proportion is also higher (0.359 against 0.231) and reading A finds the difference (p = 0.00024; reading B p = 0.11), so the absence part does not hold.

## Follower trials (Figure 8)

Every arrival at a new well (first poke of a visit) by a rat whose partner made the pair's previous
arrival is a follower trial for that rat: the partner (leader) entered an arm first and the rat then
chose (Methods, Partner behavior history analysis). It cooperates when its new well corresponds to the
partner's current well. The partner's previous choice is the outcome of the partner's most recent
follower trial, within the session (across sessions as a variant); Δt is the time from it to the
current trial. Variant: only follower trials in which the leader is still at its well when the follower
arrives (its last poke there no more than 1 s before), closer to "the follower then decides whether to
choose the matching arm".

In [ ]:
FT = []
for s in S:
    ev = []
    for side, r in s["rats"].items():
        for k in range(len(r["vw"])):
            ev.append((r["vs"][k], r["subject"], r["vw"][k], r["ve"][k]))
    ev.sort()
    cur, curend = {}, {}
    last_actor = None
    for t, subj, w, e in ev:
        other = [x for x in s["pair"] if x != subj][0]
        if last_actor == other and other in cur:
            FT.append(dict(subject=subj, partner=other, genotype=s["genotype"], pair="-".join(s["pair"]), contingency=s["contingency"], rank=s["rank"], t=t,
                           coop=int(w == cur[other]), waiting=int(curend[other] >= t - 1.0), sess=f"{'-'.join(s['pair'])}_{s['contingency']}_{s['rank']}"))
        cur[subj] = w
        curend[subj] = e
        last_actor = subj
FT = pd.DataFrame(FT).sort_values(["pair", "contingency", "rank", "t"]).reset_index(drop=True)

def add_history(FT, within=True):
    FT = FT.copy()
    FT["partner_prev"] = np.nan; FT["dt"] = np.nan; FT["own_prev1"] = np.nan; FT["own_prev2"] = np.nan
    keys = ["pair", "contingency", "rank"] if within else ["pair", "contingency"]
    for _, d in FT.groupby(keys):
        idx = d.index.values
        last = {}
        own = collections.defaultdict(list)
        for i in idx:
            subj, part = FT.at[i, "subject"], FT.at[i, "partner"]
            if part in last:
                FT.at[i, "partner_prev"] = last[part][0]; FT.at[i, "dt"] = FT.at[i, "t"] - last[part][1]
            if own[subj]:
                FT.at[i, "own_prev1"] = own[subj][-1]
            if len(own[subj]) > 1:
                FT.at[i, "own_prev2"] = own[subj][-2]
            last[subj] = (FT.at[i, "coop"], FT.at[i, "t"])
            own[subj].append(FT.at[i, "coop"])
    return FT

FTw = add_history(FT, True)
FTa = add_history(FT, False)
FTwait = add_history(FT[FT.waiting == 1].reset_index(drop=True), True)
print("fraction of follower trials in which the leader is still at its well (last poke no more than 1 s before the follower's arrival):")
print(FT.groupby(["contingency", "genotype"]).waiting.mean().round(3).to_string())
print(f"follower trials: {len(FT)}; with a partner history within the session {FTw.partner_prev.notna().sum()}, across sessions {FTa.partner_prev.notna().sum()}")
print(FTw.groupby(["contingency", "genotype"]).coop.agg(["size", "mean"]).round(3).to_string())

## fig8b-partner-history-cooperation

> After a partner's cooperative choice, WT rats are more likely to cooperate under both contingencies,
> Fmr1 rats only under 100% reward.

In [ ]:
for lab, F in (("within session", FTw), ("across sessions (variant)", FTa), ("leader waiting, within session (variant)", FTwait)):
    print(lab)
    for c in ("rc100", "rc50"):
        for g in ("WT", "Fmr1"):
            d = F[(F.contingency == c) & (F.genotype == g)].dropna(subset=["partner_prev"])
            pr = d.groupby(["subject", "partner_prev"]).coop.mean().unstack()
            w = stats.wilcoxon(pr[1.0], pr[0.0])
            print(f"  {c} {g}: P(coop | partner coop) {pr[1.0].mean():.3f}, P(coop | partner not) {pr[0.0].mean():.3f}, rats higher after cooperation {int((pr[1.0] > pr[0.0]).sum())} of {len(pr)}, Wilcoxon W = {w.statistic:.1f}, p = {w.pvalue:.4f}")

**Result: not reproduced (resting on an assumption).** With every follower trial, rats of both genotypes cooperate less after their partner cooperated, in both contingencies (for example WT at 100%, 0.362 against 0.602; 0 of 10 rats higher after cooperation in each group; Wilcoxon p = 0.002), the reverse of the claim; across sessions the same. With only trials in which the leader is still at its well, there is no effect in any group (p 0.19 to 1.0).

## fig8d-fmr1-longer-delta-t

> The time between the partner's most recent follower choice and the subject's current follower choice
> is longer in Fmr1 than in WT rats under both contingencies.

In [ ]:
for lab, F in (("all follower trials", FTw), ("leader waiting (variant)", FTwait)):
  print(lab)
  for c in ("rc100", "rc50"):
    d = F[(F.contingency == c)].dropna(subset=["dt"])
    w = d[d.genotype == "WT"].dt; f = d[d.genotype == "Fmr1"].dt
    u = stats.mannwhitneyu(f, w, alternative="two-sided")
    print(f"  {c}: Fmr1 median {f.median():.2f} s (n = {len(f)}), WT median {w.median():.2f} s (n = {len(w)}); U (Fmr1 vs WT) = {u.statistic:.0f}, p = {u.pvalue:.3g}; range {d.dt.min():.2f} to {d.dt.max():.1f} s, above 60 s: {(d.dt > 60).sum()}")
print(FTw.dropna(subset=["dt"]).groupby(["contingency", "genotype", "subject"]).dt.median().round(2).unstack(0).to_string())

**Result: reproduced (resting on an assumption).** Δt is longer in Fmr1 than in WT rats in both contingencies: medians 16.3 against 11.4 s at 100% (p = 1.5e-77) and 12.7 against 9.6 s at 50% (p = 4.9e-216). With only trials in which the leader waits, the same (32.7 against 24.7 s, p = 3.5e-37; 20.7 against 15.1 s, p = 2.6e-122). The U values (17,022,709 and 120,576,406) differ from the paper's, as the trial counts do.

## fig8e-glmm-partner-cooperation

> In a GLMM of cooperation on each follower trial, the partner's previous cooperation has a larger effect
> in WT rats at 50% (beta 0.77) than in Fmr1 rats (beta 0.13), a significant genotype interaction (beta
> 0.64); at 100% both genotypes show an effect (WT 0.36, Fmr1 0.26).

Binomial GLMM (logit), partner_prev x Δt x genotype, own history covariates, random intercepts for rat
and session, fitted by statsmodels' `BinomialBayesMixedGLM` (variational Bayes; pymer4 and R are not
in the environment). Readings: own history 1 or 2 previous follower choices; Δt in seconds or
z-scored (the seconds reading is fitted in minutes, an equivalent scaling); genotype reference WT (gives the WT beta and Fmr1 minus WT) or Fmr1 (gives the Fmr1 beta
and WT minus Fmr1). p from the posterior mean over its standard deviation.

In [ ]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
rows = []
for c in ("rc100", "rc50"):
    for own in (1, 2):
        for scale in ("s", "z"):
            d = FTw[FTw.contingency == c].dropna(subset=["partner_prev", "dt", "own_prev1"] + (["own_prev2"] if own == 2 else [])).copy()
            # seconds: fitted with Δt in minutes, the same model with Δt's coefficient rescaled (variational
            # Bayes does not converge with Δt in seconds); the partner effect at Δt = 0 is unchanged
            d["dtx"] = d.dt / 60 if scale == "s" else (d.dt - d.dt.mean()) / d.dt.std()
            for ref in ("WT", "Fmr1"):
                d["g"] = (d.genotype != ref).astype(int)
                cov = " + own_prev1" + (" + own_prev2" if own == 2 else "")
                m = BinomialBayesMixedGLM.from_formula(f"coop ~ partner_prev * dtx * g{cov}", {"rat": "0 + C(subject)", "session": "0 + C(sess)"}, d)
                fit = m.fit_vb()
                names = list(m.exog_names)
                i, j = names.index("partner_prev"), names.index("partner_prev:g")
                for term, k in (("beta_ref", i), ("interaction_other_minus_ref", j)):
                    z = fit.fe_mean[k] / fit.fe_sd[k]
                    rows.append(dict(contingency=c, own=own, dt=scale, reference=ref, term=term, beta=fit.fe_mean[k], sd=fit.fe_sd[k], p=2 * stats.norm.sf(abs(z)), n=len(d)))
t8e = pd.DataFrame(rows)
print(t8e.round(4).to_string())

**Result: not reproduced (resting on an assumption).** The partner's previous cooperation lowers the log-odds of cooperating in both genotypes and contingencies under every reading (WT beta -0.81 to -1.43, Fmr1 -0.92 to -1.43, all p below 0.0001). At 50% the genotype difference is near zero with Δt unscaled (p = 0.66 to 0.99); z-scored, WT minus Fmr1 is +0.14 to +0.15 (p below 0.0001), positive as claimed but far from 0.64, while both betas are negative.

## Asynchronous trials and gaze (Figure 7C to 7H)

An asynchronous trial (Methods, Quantification of peer-directed head orientation events): the two
rats are at distinct, noncorresponding wells (each within a visit, first to last poke); the follower
is the rat that departs first (its visit ends) while the partner is still at its well. The trial runs
from the moment both are at those wells to the follower's last poke there. The outcome is a match when
one of the dataset's matched pokes falls at the leader's well between the follower's departure and the
end of its next visit. Trials shorter than 0.5 s (too few frames for quartiles) are dropped.
Gaze per frame from the processed neck, head and body positions (cm, one video frame for both rats):
bearing angle of Eq. 11 below 40 degrees. Follower-only gaze: the follower is oriented to the leader
and the leader is not oriented to the follower; mutual: both.

In [ ]:
TRI = []
dropped_short = 0
for s in S:
    rats = s["rats"]
    sides = ("left", "right")
    for fs in sides:
        ls = "right" if fs == "left" else "left"
        F, L = rats[fs], rats[ls]
        for k in range(len(F["vw"])):
            wf, sf, ef = F["vw"][k], F["vs"][k], F["ve"][k]
            j = np.flatnonzero((L["vs"] <= ef) & (L["ve"] > ef) & (L["vw"] != wf))
            if len(j) == 0:
                continue
            j = j[0]
            start = max(sf, L["vs"][j])
            if ef - start < 0.5:
                dropped_short += 1
                continue
            wl = L["vw"][j]
            nxt_end = F["ve"][k + 1] if k + 1 < len(F["vw"]) else ef
            match = bool(np.any((s["match_w"] == wl) & (s["match_t"] > ef) & (s["match_t"] <= nxt_end + 0.01)))
            # gaze
            iF0, iF1 = np.searchsorted(F["pos_t"], [start, ef])
            tt = F["pos_t"][iF0:iF1]
            if len(tt) < 4:
                dropped_short += 1
                continue
            iL = np.clip(np.searchsorted(L["pos_t"], tt), 0, len(L["pos_t"]) - 1)
            thF = coop.bearing(F["pos"]["neck"][iF0:iF1], F["pos"]["head"][iF0:iF1], L["pos"]["body"][iL])
            thL = coop.bearing(L["pos"]["neck"][iL], L["pos"]["head"][iL], F["pos"]["body"][iF0:iF1])
            ok = np.isfinite(thF) & np.isfinite(thL)
            fo = (thF < 40) & (thL >= 40)
            mu = (thF < 40) & (thL < 40)
            q = np.minimum((np.arange(len(tt)) * 4) // len(tt), 3)
            row = dict(follower=F["subject"], leader=L["subject"], genotype=s["genotype"], pair="-".join(s["pair"]), contingency=s["contingency"], rank=s["rank"],
                       sess=f"{'-'.join(s['pair'])}_{s['contingency']}_{s['rank']}", start=start, end=ef, duration=ef - start, match=int(match),
                       valid=ok.mean(), gF=fo[ok].mean() if ok.any() else np.nan, gM=mu[ok].mean() if ok.any() else np.nan)
            for qq in range(4):
                m = ok & (q == qq)
                row[f"gF_q{qq + 1}"] = fo[m].mean() if m.any() else np.nan
                row[f"gM_q{qq + 1}"] = mu[m].mean() if m.any() else np.nan
            TRI.append(row)
TRI = pd.DataFrame(TRI)
print(f"asynchronous trials: {len(TRI)}; dropped as shorter than 0.5 s or 4 frames: {dropped_short}")
print(TRI.groupby(["contingency", "genotype"]).agg(n=("match", "size"), match=("match", "mean"), duration=("duration", "median"), gF=("gF", "mean"), gM=("gM", "mean"), valid=("valid", "mean")).round(3).to_string())
print(f"trial duration {TRI.duration.min():.2f} to {TRI.duration.max():.1f} s; frames with both bearings defined {TRI.valid.mean():.3f}")

## fig7c-trial-duration-vs-performance

> Session mean trial duration is negatively correlated with cooperative performance in both genotypes
> (WT R = -0.51, Fmr1 R = -0.40).

Readings of the trial: (a) the asynchronous trial above; (b) the interval between consecutive
matches; (c) the interval between consecutive transitions of the pair. Points: one per pair and
session, or one per rat and session (for (a), the rat as follower; for (c), the rat's own
transitions; (b) is a pair measure). 100% and 50% sessions combined.

In [ ]:
perf = P.set_index(["unit", "contingency", "rank"]).performance
rows = []
pa = TRI.groupby(["pair", "contingency", "rank", "genotype"]).duration.mean().reset_index()
pb, pc, ra, rc = [], [], [], []
for s in S:
    key = ("-".join(s["pair"]), s["contingency"], s["rank"])
    mt = np.sort(s["match_t"])
    pb.append(dict(pair=key[0], contingency=key[1], rank=key[2], genotype=s["genotype"], duration=np.diff(mt).mean() if len(mt) > 1 else np.nan))
    allv = np.sort(np.concatenate([r["vs"][1:] for r in s["rats"].values()]))
    pc.append(dict(pair=key[0], contingency=key[1], rank=key[2], genotype=s["genotype"], duration=np.diff(allv).mean() if len(allv) > 1 else np.nan))
    for r in s["rats"].values():
        rc.append(dict(pair=key[0], contingency=key[1], rank=key[2], genotype=s["genotype"], duration=np.diff(r["vs"]).mean() if len(r["vs"]) > 1 else np.nan))
ra = TRI.groupby(["follower", "pair", "contingency", "rank", "genotype"]).duration.mean().reset_index()
for reading, pts, d in (("a", "pair", pa), ("a", "rat", ra), ("b", "pair", pd.DataFrame(pb)), ("c", "pair", pd.DataFrame(pc)), ("c", "rat", pd.DataFrame(rc))):
    d = d.copy()
    d["performance"] = [perf.get((p, c, k), np.nan) for p, c, k in zip(d.pair, d.contingency, d["rank"])]
    d = d.dropna(subset=["duration", "performance"])
    for g in ("WT", "Fmr1"):
        dd = d[d.genotype == g]
        r, p = stats.pearsonr(dd.duration, dd.performance)
        rows.append(dict(trial=reading, points=pts, genotype=g, n=len(dd), R=r, p=p, in_tol=abs(r - (-0.51 if g == "WT" else -0.40)) <= (0.051 if g == "WT" else 0.04)))
t7c = pd.DataFrame(rows)
print(t7c.round(4).to_string())

**Result: partially reproduced.** Session mean trial duration and performance are negatively correlated in both genotypes under every reading (R from -0.31 to -0.70, all p below 0.0001). No reading puts both R within tolerance: with asynchronous trials and one point per pair and session R is -0.39 (WT) and -0.51 (Fmr1), per rat and session -0.31 and -0.42 (only Fmr1 within tolerance).

## fig7e-gaze-final-quartile

> Follower-only gaze toward the leader increases from the first to the later quartiles of a trial and is
> greatest just before departure; mutual gaze shows no quartile effect, and there is no genotype effect.

Mean per rat (as follower) and quartile over its asynchronous trials of all analysed sessions of both
contingencies; mixed ANOVA (genotype between, quartile within, rat as subject); Holm-corrected
paired t tests of quartile 1 against 2, 3 and 4.

In [ ]:
import pingouin as pg
G = []
for (rat, g), d in TRI.groupby(["follower", "genotype"]):
    for qq in range(1, 5):
        G.append(dict(rat=rat, genotype=g, quartile=qq, gF=d[f"gF_q{qq}"].mean(), gM=d[f"gM_q{qq}"].mean()))
G = pd.DataFrame(G)
print(G.groupby(["genotype", "quartile"])[["gF", "gM"]].mean().round(4).to_string())
for dv in ("gF", "gM"):
    a = pg.mixed_anova(data=G, dv=dv, within="quartile", between="genotype", subject="rat", effsize="np2")
    print(dv); print(a.round(5).to_string())
ph = pg.pairwise_tests(data=G, dv="gF", within="quartile", subject="rat", padjust="holm")
print(ph[["A", "B", "T", "p_unc" if "p_unc" in ph else "p-unc", "p_corr" if "p_corr" in ph else "p-corr"]].round(5).to_string())
print("quartile with the highest follower-only gaze, per genotype:", G.groupby(["genotype", "quartile"]).gF.mean().groupby("genotype").idxmax().to_dict())

**Result: reproduced (resting on an assumption).** Follower-only gaze rises over the trial in both genotypes (Fmr1 0.085 to 0.139, WT 0.119 to 0.156), the quartile effect is F(3,54) = 19.3, p below 0.001 (Greenhouse-Geisser p = 0.00006, partial eta squared 0.52 against the paper's 0.40), quartile 4 is highest in both genotypes, and the Holm-corrected tests put quartile 1 below quartiles 2, 3 and 4 (p = 0.00007, 0.0006, 0.0006). Mutual gaze has no quartile effect (F(3,54) = 1.26, p = 0.30). Genotype and its interactions are not significant (p = 0.22, 0.58; mutual 0.78, 0.95). The trials rest on this attempt's reading of the asynchronous trial.

## fig7g-gaze-100-late and fig7h-gaze-50-fmr1-follower

Binomial GLMM of match on each asynchronous trial, logit(P(match)) ~ (gF + gM) x genotype x block,
random intercepts for the follower rat and the session (Eq. 12), fitted by variational Bayes
(`BinomialBayesMixedGLM`; glmmTMB is not in the environment). The model is written so that each
Fmr1 minus WT gaze slope in a block is one coefficient (the between-genotype contrast of emtrends in
that block; with one contrast per block the Tukey adjustment changes nothing). Readings: blocks are
thirds of the sessions (100%: 7, 7, 7; 50%: 12, 12, 11 or 11, 12, 12; every pair has the same
session count, so thirds by the pair's own count are the same); gF and gM as fractions of the trial,
or z-scored.

In [ ]:
def gaze_glmm(c, cuts, scale):
    d = TRI[TRI.contingency == c].dropna(subset=["gF", "gM"]).copy()
    d["block"] = np.select([d["rank"] <= cuts[0], d["rank"] <= cuts[1]], [1, 2], 3)
    d["g"] = (d.genotype == "Fmr1").astype(int)
    for v in ("gF", "gM"):
        d[v + "x"] = d[v] if scale == "fraction" else (d[v] - d[v].mean()) / d[v].std()
    f = "match ~ C(block) * g + gFx:C(block) + gFx:C(block):g + gMx:C(block) + gMx:C(block):g"
    m = BinomialBayesMixedGLM.from_formula(f, {"rat": "0 + C(follower)", "session": "0 + C(sess)"}, d)
    fit = m.fit_vb()
    out = []
    for v in ("gF", "gM"):
        for b in (1, 2, 3):
            name = f"{v}x:C(block)[{b}]:g"
            k = m.exog_names.index(name)
            z = fit.fe_mean[k] / fit.fe_sd[k]
            out.append(dict(contingency=c, blocks="-".join(map(str, cuts)), scale=scale, gaze=v, block=["early", "mid", "late"][b - 1],
                            contrast=fit.fe_mean[k], se=fit.fe_sd[k], p=2 * stats.norm.sf(abs(z)), n=len(d)))
    return out

In [ ]:
t7g = pd.DataFrame([r for scale in ("fraction", "z") for r in gaze_glmm("rc100", (7, 14), scale)])
print(t7g.round(4).to_string())

**Result: not reproduced (resting on an assumption).** At 100% the late Fmr1 minus WT contrast for follower-only gaze is negative (-0.57 as a fraction, p = 0.11; -0.13 z-scored, p = 0.14), and for mutual gaze it is small and not significant (0.19, p = 0.73; 0.03, p = 0.72). The early and mid follower-only contrasts are not significant (p 0.07 to 0.77).

In [ ]:
t7h = pd.DataFrame([r for cuts in ((12, 24), (11, 23)) for scale in ("fraction", "z") for r in gaze_glmm("rc50", cuts, scale)])
print(t7h.round(5).to_string())

**Result: not reproduced (resting on an assumption).** At 50% only the early follower-only contrast is positive and significant (0.48 to 0.49 as a fraction, p = 0.016 and 0.017; 0.11 z-scored, p = 0.029 and 0.030). The mid and late contrasts are negative under every block and scaling reading (-0.06 to -0.43). The mutual-gaze contrast is significant early under the 11-12-12 blocks (p = 0.003).

## Blocked claims

- figs1-mixed-pairs-impaired, fig2c-opaque-barrier-mixed-pairs: the dandiset holds no mixed WT-Fmr1 pair.
  Its 20 subjects form 10 pairs, each of one genotype (data check above).
- fig2b-opaque-barrier-lowers-performance, fig2c-opaque-barrier-mixed-pairs, and the sensory-deprived part
  of figs2-jaccard-similarity: no session is marked as sensory deprived. Every file's description names
  only the 100% or the 50% contingency (data check), and the files carry no barrier or noise field.
- fig2e-visual-cue-learning-similar, fig2f-cue-latency-wt-shorter: the visual-cue association task (single
  rats, LED cues) is not in the dandiset.
- fig3g-virtual-agent-performance-similar, fig3h-virtual-agent-efficiency-similar: the virtual-agent task
  (single rats) is not in the dandiset.
- fig6-marl-partner-aware: a simulation that uses no recorded data, with no published code; the learning
  rate, discount factor, timesteps per episode, number of runs, state encoding and the definition of
  "coordination efficiency (% of optimal matches)" are not given, and the reported 70% depends on them.
  Not run.
- figs3-leadership-history: blocked by its criterion until the supplement is read; the readings it names
  are shown above, not judged.

In [ ]:
print(f"notebook run time: {(time.time() - T_START) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig1e-match-rate-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig1f-transition-rate-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig1g-match-transition-regression | partially reproduced (assumption) |
| fig1h-above-chance-late | partially reproduced (assumption) |
| fig1i-performance-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig1j-partner-location-predicts-choice | not reproduced (assumption) |
| figs1-mixed-pairs-impaired | blocked, data missing |
| fig2b-opaque-barrier-lowers-performance | blocked, data missing |
| fig2c-opaque-barrier-mixed-pairs | blocked, data missing |
| fig2e-visual-cue-learning-similar | blocked, data missing |
| fig2f-cue-latency-wt-shorter | blocked, data missing |
| fig3c-triplets-by-contingency | partially reproduced (assumption) |
| fig3d-entropy-wt-lower | partially reproduced (assumption) |
| fig3e-exploration-efficiency-wt-higher | blocked, methods underspecified (readings disagree) |
| fig3g-virtual-agent-performance-similar | blocked, data missing |
| fig3h-virtual-agent-efficiency-similar | blocked, data missing |
| figs2-jaccard-similarity | not reproduced (assumption) |
| fig4b-arrival-lag-wt-shorter | not reproduced |
| fig4c-departure-lag-wt-shorter | not reproduced |
| fig4d-dwell-time | reproduced |
| fig4eg-cross-correlation-wt-higher | partially reproduced |
| fig4h-peak-lag-distribution | not reproduced (assumption) |
| fig4i-peak-correlation-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig5d-leading-near-half | reproduced |
| fig5e-leading-difference-50 | not reproduced |
| figs3-leadership-history | blocked, methods underspecified (by the claims file) |
| fig6-marl-partner-aware | blocked, methods underspecified |
| fig7b-fmr1-more-asynchronous-matches | partially reproduced (assumption) |
| fig7c-trial-duration-vs-performance | partially reproduced |
| fig7e-gaze-final-quartile | reproduced (assumption) |
| fig7g-gaze-100-late | not reproduced (assumption) |
| fig7h-gaze-50-fmr1-follower | not reproduced (assumption) |
| fig8b-partner-history-cooperation | not reproduced (assumption) |
| fig8d-fmr1-longer-delta-t | reproduced (assumption) |
| fig8e-glmm-partner-cooperation | not reproduced (assumption) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The paper's analysis code is not published, so every event definition is this attempt's reading of the Methods: a visit is a run of pokes at one well; synchronous and asynchronous matches, asynchronous gaze trials and follower trials are defined from the poke sequences as stated in their sections.
- Pokes within 0.5 s of a poke at another well by the same rat are dropped as sensor crosstalk (an assumption; 4.8% of pokes, 140 of 1,120 rat sessions changed). The raw sequences are shown for transitions and performance.
- The ANOVA the legends call "Two-way RM ANOVA" has df that fit an ordinary two-way ANOVA with sessions as replicates (reading A). With pairs or rats as subjects (reading B) most genotype effects are not significant. The claims file blocks a claim where the two disagree, which decides four claims.
- The permutation test of Figure 1H uses matches rebuilt from the pokes (6 s hold, no match at the previous match's well), because the dataset's matches cannot be recomputed under a shift.
- The 1,000-shuffle null of Figure 1J uses the class a one-hot logistic regression predicts (the modal next well per partner well), which agrees with scikit-learn on 98% of real folds; the real accuracies are scikit-learn's.
- The GLMMs are fitted by variational Bayes in statsmodels (no R, glmmTMB or pymer4 in the environment).

What was found about the dataset:

- It holds the 10 same-genotype pairs (5 WT, 5 Fmr1) under 100% and 50% reward. The mixed pairs, the sensory-deprived sessions, the visual-cue task and the virtual-agent task are not in it.
- About 4.8% of pokes are near-simultaneous pokes at two wells by one rat (up to 128,979 pokes and 2,447 transitions in one rat session), which inflate transition counts unless removed.
- The matched pokes never repeat the well of the previous match (0 of 19,602), a rule the paper does not state for the rat task; poke offsets are not recorded.
- Session numbers have gaps for three pairs, and sessions run from 10.7 to 57.2 min against the paper's 20 to 30 min; 6.6% of rat sessions have pokes outside the session epoch.
- Both files of a pair session hold the same events; each recording was counted once. Positions and pokes share a clock (head 2.7 cm from the poked well at the poke, 30 to 46 cm when shifted by 5 to 10 s).